# Value Score Engineering

The goal of this notebook is to create position-adjusted EPA-based offensive value scores for NFL quarterbacks, running backs, wide receivers, and tight ends.

Player value is difficult to measure directly, so this project uses EPA per game as the primary value metric. EPA (*Expected Points Added*) is useful because it captures the expected point impact of a player's plays rather than only counting raw box-score production. EPA works by using historical data to assign an "expected points" value to the specific game situation, considering the down, distance to the first marker, yard line, and time remaining. EPA is then calculated as Expected Points (After the Play) - Expected Points (Before the Play).

For quarterbacks, the value metric uses passing plus rushing EPA per game. For running backs, wide receivers, and tight ends, the value metric uses rushing plus receiving EPA per game. Scores are standardized within each season-position group so that players are compared to others at the same position in the same season.

## Load Cleaned Player-Season Data

This section loads the cleaned player-season dataset created in the previous notebook. Each row represents one player-season for a quarterback, running back, wide receiver, or tight end.

In [ ]:
from pathlib import Path

import pandas as pd
import numpy as np

def find_project_root(expected_file):
    """Find the repo root from common VS Code/Jupyter working directories."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / expected_file).exists():
            return candidate
    raise FileNotFoundError(
        f"Could not find {expected_file} from working directory {Path.cwd()}"
    )

project_root = find_project_root("data/processed/skill_player_seasons_2016_2025.csv")
processed_dir = project_root / "data" / "processed"

skill_season_path = processed_dir / "skill_player_seasons_2016_2025.csv"
skill_season = pd.read_csv(skill_season_path)

print(skill_season.shape)
skill_season.head()


## Create Final Supporting Features

The main value score will be based on EPA per game, but I also create several supporting features that will be useful for interpreting player rankings later.

For non-quarterbacks, touchdowns per game help describe scoring production. For quarterbacks, interceptions per game help describe negative plays. These variables are not the main value score, but they will be useful in later analysis.

In [ ]:
# Keep only players with at least one game played
skill_season = skill_season[skill_season["games_played"] > 0].copy()

# Non-QB scoring feature
skill_season["scrimmage_tds_per_game"] = (
    skill_season["scrimmage_tds"] / skill_season["games_played"]
)

# QB negative-play feature
skill_season["interceptions_per_game"] = (
    skill_season["passing_interceptions"] / skill_season["games_played"]
)

# EPA-only benchmark score variable for all players
skill_season["epa_only_score"] = skill_season["epa_per_game"]

skill_season.head()


## Filter Out Small-Sample Players

Players with fewer than four games played were removed from the value score dataset to reduce noise from very small samples.

In [ ]:
# Filter out very small sample players
value_df = skill_season[skill_season["games_played"] >= 4].copy()

print(value_df.shape)
value_df.head()

## Define a Group Z-Score Function

The value score should compare players to others at the same position in the same season. This is important because raw EPA differs across positions, especially between quarterbacks and non-quarterbacks.

This function creates z-scores within season-position groups. A z-score of 2 means the player was about two standard deviations above the average player at that position in that season.

In [ ]:
def add_group_zscore(df, cols, group_cols=["season", "position"]):
    """
    Add z-score columns for selected variables within season-position groups.
    This compares each player to others at the same position in the same season.
    """
    df = df.copy()
    
    for col in cols:
        z_col = f"{col}_z"
        group_mean = df.groupby(group_cols)[col].transform("mean")
        group_std = df.groupby(group_cols)[col].transform("std")
        
        df[z_col] = (df[col] - group_mean) / group_std
    
    return df

## Create EPA-Based Value Scores

The main value score is based on EPA per game, standardized within each season-position group.

For quarterbacks, I use `qb_epa_per_game`, which combines passing and rushing EPA.

For running backs, wide receivers, and tight ends, I use `scrimmage_epa_per_game`, which combines rushing and receiving EPA.

This keeps the main score simple, transparent, and less dependent on subjective weighting choices.

In [ ]:
# Split QBs and non-QBs
non_qb_positions = ["RB", "WR", "TE"]

non_qb = value_df[value_df["position"].isin(non_qb_positions)].copy()
qbs = value_df[value_df["position"] == "QB"].copy()

# Non-QB EPA-only value score
non_qb = non_qb.dropna(subset=["scrimmage_epa_per_game"])
non_qb = add_group_zscore(non_qb, ["scrimmage_epa_per_game"])

non_qb["value_score"] = non_qb["scrimmage_epa_per_game_z"]
non_qb["value_epa_per_game"] = non_qb["scrimmage_epa_per_game"]
non_qb["value_metric"] = "position_adjusted_epa_per_game"

# QB EPA-only value score
qbs = qbs.dropna(subset=["qb_epa_per_game"])
qbs = add_group_zscore(qbs, ["qb_epa_per_game"])

qbs["value_score"] = qbs["qb_epa_per_game_z"]
qbs["value_epa_per_game"] = qbs["qb_epa_per_game"]
qbs["value_metric"] = "position_adjusted_epa_per_game"


## Combine Quarterbacks and Non-Quarterbacks

After creating separate EPA-based value scores for quarterbacks and non-quarterbacks, I combine the datasets back into one value-scored player-season dataset.

In [ ]:
value_scored = pd.concat([qbs, non_qb], ignore_index=True)

print(value_scored.shape)
value_scored.head()

## Add Position-Season Ranks and Percentiles

The raw value score is useful, but ranks and percentiles make the results easier to interpret.

The rank compares players within the same season and position. The percentile gives a relative standing, where values closer to 1 indicate stronger performance within that season-position group.

In [ ]:
value_scored["position_season_rank"] = (
    value_scored
    .groupby(["season", "position"])["value_score"]
    .rank(ascending=False, method="min")
)

value_scored["position_season_percentile"] = (
    value_scored
    .groupby(["season", "position"])["value_score"]
    .rank(pct=True)
)

value_scored[
    ["season", "player_display_name", "position", "team", 
     "games_played", "value_score", "position_season_rank", 
     "position_season_percentile"]
].sort_values(["season", "position", "position_season_rank"]).head(30)

## Sanity Check Player Rankings

To make sure the value score is reasonable, I check the top players at each position for a recent season. The goal is not for the ranking to perfectly match public opinion, but the top players should generally make football sense.

If unusual low-volume players appear too highly, the games-played cutoff or additional filters may need to be revisited.

In [ ]:
season_to_check = 2024

for pos in ["QB", "RB", "WR", "TE"]:
    print()
    print("Top", pos + "s", "in", season_to_check)
    display(
        value_scored[
            (value_scored["season"] == season_to_check) &
            (value_scored["position"] == pos)
        ][
            ["player_display_name", "team", "games_played", 
             "value_epa_per_game", "value_score", "position_season_rank",
             "position_season_percentile"]
        ]
        .sort_values("value_score", ascending=False)
        .head(10)
    )


## Inspect Bottom Rankings

I also inspect the lowest-ranked players to check whether the bottom of the distribution is reasonable. This can reveal issues with missing data, low-volume players, or unusual position assignments.

In [ ]:
season_to_check = 2024

for pos in ["QB", "RB", "WR", "TE"]:
    print()
    print("Bottom", pos + "s", "in", season_to_check)
    display(
        value_scored[
            (value_scored["season"] == season_to_check) &
            (value_scored["position"] == pos)
        ][
            ["player_display_name", "team", "games_played", 
             "value_epa_per_game", "value_score", "position_season_rank",
             "position_season_percentile"]
        ]
        .sort_values("value_score", ascending=True)
        .head(10)
    )


## Save Player Value Scores

This saves the value-scored player-season dataset locally for exploratory analysis, predictive modeling, and later salary-efficiency analysis. The processed CSV is intentionally ignored by Git.

In [ ]:
output_path = processed_dir / "player_value_scores_2016_2025.csv"

value_scored.to_csv(output_path, index=False)

print(f"Saved {len(value_scored):,} rows to {output_path}")
